# Setup & Installation

- `faiss-cpu:` For fast similarity search (finding the relevant movies).
- `sentence-transformers:` To create the embeddings (converting text to numbers).
- `openai:` To connect to OpenRouter (OpenRouter is compatible with the OpenAI SDK).
- `rank_bm25:` allows you to rank documents by relevance

In [68]:
! pip install pandas sentence-transformers faiss-cpu openai
! pip install rank_bm25

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


# Import

In [69]:
import os
import pandas as pd
import numpy as np
import re
import faiss
import pickle

from sentence_transformers import SentenceTransformer
from openai import OpenAI
from rank_bm25 import BM25Okapi
from tqdm import tqdm
from sentence_transformers import CrossEncoder

## Load the data

In [70]:
movies = pd.read_csv('/kaggle/input/all-data-movies/final_merged_movies.csv')
print(movies.shape)

(4801, 23)


In [71]:
movies.columns

Index(['budget', 'homepage', 'id', 'original_language', 'original_title',
       'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'vote_average', 'vote_count',
       'Poster', 'on_netflix', 'on_disney', 'on_amazon', 'on_hulu',
       'search_text'],
      dtype='object')

# Configuration (OpenRouter)

In [72]:
mykey = "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" 
#mykey=""

In [73]:
# 1. Setup client
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY", mykey),
)

## Check the Connection:

In [111]:
try:
    # 2. Try a simple completion
    response = client.chat.completions.create(
        model="arcee-ai/trinity-large-preview:free",
        messages=[
            {"role": "user", "content": "Say 'Connection Successful!' if you can hear me."}
        ],
        # Optional: OpenRouter headers for ranking/attribution
        extra_headers={
            "HTTP-Referer": "http://localhost:3000", 
            "X-Title": "Connection Test Script",
        }
    )

    # 3. Print the result
    print("--- OpenRouter Status ---")
    print(f"Message: {response.choices[0].message.content}")
    print(f"Model used: {response.model}")

except Exception as e:
    print(f"Connection failed: {e}")

--- OpenRouter Status ---
Message: Connection Successful!
Model used: arcee-ai/trinity-large-preview:free


# Embeddings & vectior dataset

## Embedding

In [75]:
print("Initializing Embedding Model")
embedder = SentenceTransformer('all-mpnet-base-v2')

Initializing Embedding Model


In [76]:
movie_vectors = np.load('/kaggle/input/embeddings-and-vectior-dataset-files/movie_vectors.npy')

## FAISS

## BM25

## FAISS and BM25

In [77]:
# Define file paths
FAISS_PATH = "/kaggle/input/embeddings-and-vectior-dataset-files/movie_faiss.index"
BM25_PATH = "/kaggle/input/embeddings-and-vectior-dataset-files/bm25_model.pkl"

if os.path.exists(FAISS_PATH) and os.path.exists(BM25_PATH):
    print("--- Loading existing indexes from disk ---")
    index = faiss.read_index(FAISS_PATH)
    with open(BM25_PATH, 'rb') as f:
        bm25 = pickle.load(f)
else:
    print("--- Creating new indexes (this may take a moment) ---")
    
    # 1. Build FAISS
    dimension = movie_vectors.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(np.array(movie_vectors))
    faiss.write_index(index, FAISS_PATH)
    
    # 2. Build BM25
    tokenized_corpus = [doc.split(" ") for doc in movies['search_text']]
    bm25 = BM25Okapi(tokenized_corpus)
    with open(BM25_PATH, 'wb') as f:
        pickle.dump(bm25, f)

print("Setup Complete! Ready for retrieval.")

--- Loading existing indexes from disk ---
Setup Complete! Ready for retrieval.


## Reranking 

In [78]:
# Reranking model is trained to judge exactly how relevant a search result is
print("Load the Reranking Model")
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("Indexing Complete!")

Load the Reranking Model
Indexing Complete!


# Search movies function

In [79]:
def search_movies(query, k=5):
    """
    Hybrid Search (Vector + BM25) + Reranking
    Guaranteed to return a list (never None).
    """
    try:
        # If query is empty, return an empty list
        if not query or not isinstance(query, str):
            print("⚠️ Warning: Empty query in search_movies")
            return []

        # --- A. VECTOR SEARCH ---
        try:
            query_vector = embedder.encode([query]) # Sentence-Transformers
            distances, indices = index.search(query_vector, k=20) # will return 20 movie that Nearest Neighbors
            vector_candidates = [idx for idx in indices[0] if idx >= 0] # filter result, will use valuse that not -1 
        except Exception as e:
            print(f"⚠️ Vector Search failed: {e}")
            vector_candidates = []

        # --- B. BM25 SEARCH ---
        try:
            tokenized_query = query.split(" ")
            bm25_candidates_docs = bm25.get_top_n(tokenized_query, movies['search_text'].tolist(), n=20)
            
            bm25_indices = []
            for doc in bm25_candidates_docs:
                matches = movies.index[movies['search_text'] == doc].tolist()
                if matches:
                    bm25_indices.append(matches[0])
        except Exception as e:
            print(f"⚠️ BM25 Search failed: {e}")
            bm25_indices = []

        # --- C. Hybrid Search: COMBINE ---
        all_candidate_indices = list(set(vector_candidates + bm25_indices))
        
        if not all_candidate_indices:
            print("⚠️ No candidates found.")
            return []

        candidate_movies = []
        candidate_pairs = []
        
        for idx in all_candidate_indices:
            try:
                row = movies.loc[idx]
                candidate_movies.append(row)
                candidate_pairs.append([query, row['search_text']])
            except:
                continue

        if not candidate_pairs:
            return []

        # --- D. RERANK ---
        try:
            scores = reranker.predict(candidate_pairs)
            results_with_scores = list(zip(candidate_movies, scores)) # movies with the score
            results_with_scores.sort(key=lambda x: x[1], reverse=True) # sort the movies
            return [item[0] for item in results_with_scores[:k]] #return the top
        except Exception as e:
            print(f"⚠️ Reranker failed: {e}")
            # Fallback: Just return the vector results without sorting
            return [movies.loc[i] for i in vector_candidates[:k]]

    except Exception as e:
        print(f"❌ Critical Error in search_movies: {e}")
        return [] # Always return a list, never None

In [80]:
results = search_movies("movies on disney with robot", k=3)

for movie in results:
    print(f"Movie: {movie['original_title']}")
    print(f"Rating: {movie['vote_average']}")
    print(f"Watch on Netflix: {movie['on_netflix']}")
    print("-" * 20)

Movie: Big Hero 6
Rating: 7.8
Watch on Netflix: False
--------------------
Movie: The Black Hole
Rating: 6.1
Watch on Netflix: False
--------------------
Movie: Robots
Rating: 6.0
Watch on Netflix: False
--------------------


In [81]:
search_movies("")

⚠️ Warning: Empty query in search_movies


[]

## RETRIEVAL & RAG LOGIC

## Rewrite function

In [163]:
def rewrite_query(user_message, history):
    
    refined_query = user_message 
    
    try:

        chat_context = ""
        
        # Inside rewrite_query loop:
        for turn in history[-5:]:
            if isinstance(turn, dict):
                # Gradio often uses 'role' and 'content'
                role = turn.get('role')
                content = turn.get('content', '')
                if role == 'user':
                    chat_context += f"User: {content}\n"
                else:
                    chat_context += f"AI: {content}\n"
            
        # --- THE FIX: Explicitly tell it to resolve "It/That/Him" ---
        prompt = f"""
        You are a Search Query Optimizer for a movie database.
        
        Your task is to rewrite the user's message into a short, precise English search query.
        
        Rules:
        1. OUTPUT LANGUAGE: English only.
        2. TITLE & PLOT: Extract explicit movie titles and strong plot descriptors.
        3. CONTEXT RESOLUTION: Resolve pronouns (it, that, him, her, they) using the provided History.
        4. GENRE MAPPING:
            - kids → Family
            - scary → Horror
            - cartoon → Animation
        5. PLATFORM HANDLING:
            - Detect any streaming platform (Netflix, Disney, Amazon Prime Video, etc.).
            - If a platform appears in History but NOT in the current Question, carry it forward into the query.
            - If no platform appears in either History or Question, do NOT add one.
        6. ATMOSPHERE & CHARACTERS: 
            - Identify data from the poster description.
            - If the user mentions visual cues (colors, lighting, mood, character poses), explicitly include these as keywords (e.g., "red background", "silhouette", "raining").
        7. NO HALLUCINATION: Do not invent titles, platforms, or details.
        8. FORMAT: Return ONLY a single-line search query string. Append the platform at the end if present.
        
        Inputs:
        History: {chat_context}
        User Message: "{user_message}"
        
        Return only the optimized search query.
"""

        response = client.chat.completions.create(
            model="arcee-ai/trinity-large-preview:free",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=50,
            temperature=0, # Lower temperature = more focus
        )

        refined_query = response.choices[0].message.content.strip().replace('"', '')
        print(f"✨ AI Optimized Query: {refined_query}")

    except Exception as e:
        print(f"⚠️ AI Rewrite failed: {e}")
    
    return refined_query

**Check the rewrite_query function:**

In [159]:
# --- STEP 1:  History ---
fake_history = [
    ("I'm looking for a kids movie not on Amazon","a kids movie not from Amazon")
]

user_msg = "with a rat"

# --- STEP 3: Run ---
# The AI should now combine these to: "Family movie with a rat" (or Ratatouille)
results = rewrite_query(user_msg, fake_history)

✨ AI Optimized Query: movie with a rat not on Amazon


In [160]:
# --- STEP 1:  History ---
fake_history = [
    ("I'm looking for a kids movie on disney", "disney movie")
]

# --- STEP 2: The follow-up ---
user_msg = "with a rat"

# --- STEP 3: Run ---
# The AI should now combine these to: "Family movie with a rat" (or Ratatouille)
results = rewrite_query(user_msg, fake_history)

✨ AI Optimized Query: rat movie disney


## The chat

In [172]:
def chat_logic(user_message, history=[]):
    try:
        
        # 1. Clean history of HTML tags before rewriting
        
        clean_history = []
        for h in history:
            # h[0] is user, h[1] is AI
            clean_ai_msg = re.sub('<[^<]+?>', '', h[1]) 
            clean_history.append((h[0], clean_ai_msg))

        # 2. Use the cleaned history for the rewriter
        recent_history = clean_history[-5:] 
        optimized_query = rewrite_query(user_message, recent_history)

        # 3. RETRIEVE: Search using the AI-optimized string
        # Model Tip: Ensure your 'search_movies' uses a hybrid search (semantic + keyword)
        retrieved_movies = search_movies(optimized_query, k=5)

        context_str = ""
        if retrieved_movies:
            context_str = "\n".join([f"- {m.get('search_text', 'N/A')}" for m in retrieved_movies])
        else:
            context_str = "No specific matches found."

        # 4. GENERATE: Conversational Response
        # Using a slightly higher temperature (0.5) for a "friendlier" feel
        response = client.chat.completions.create(
            model="arcee-ai/trinity-large-preview:free", 
            messages=[
                {"role": "system", "content": "You are a friendly Movie Assistant. Use the provided data to recommend films. Be brief (2-3 sentences)."},
                {"role": "user", "content": f"Context:\n{context_str}\n\nQuestion: {user_message}"}
            ],
            temperature=0.5,
            max_tokens=150
        )
        
        answer = response.choices[0].message.content.strip()

        return answer,retrieved_movies

    except Exception as e:
        return f"I ran into a snag: {str(e)}"

In [173]:
chat_logic("movie about a rat")

✨ AI Optimized Query: movie about a rat


('"Ratatouille" is a delightful animated film about a rat named Remy who dreams of becoming a great French chef. Despite the challenges of being a rat in a rodent-phobic profession, Remy\'s passion for cooking leads to a hilarious and exciting adventure that turns the culinary world of Paris upside down.',
 [budget                                                           48000000
  homepage                                                              NaN
  id                                                                   9896
  original_language                                                      en
  original_title                                                   Rat Race
  overview                In an ensemble film about easy money, greed, m...
  popularity                                                       18.60049
  production_companies    [{"name": "Paramount Pictures", "id": 4}, {"na...
  production_countries    [{"iso_3166_1": "CA", "name": "Canada"}, {"iso...
  releas

# RAG evaluation

## Retrieval Metrics (Recall@k & MRR)

In [139]:
def evaluate_retrieval(test_cases, k=5):
    """
    Evaluates the search engine using Recall@K and MRR (Mean Reciprocal Rank).
    """
    recalls = []
    reciprocal_ranks = []

    print(f"🔎 Starting Evaluation (Top-{k} results)...")
    print("-" * 50)
    
    for case in test_cases:
        query = case["query"]
        expected_title = case["expected_title"]
        
        # 1. Execute the search
        # We use search_movies (your core retrieval function)
        results = search_movies(query, k=k)
        
        # 2. Extract titles for comparison
        # Using .get() to avoid crashes if a key is missing
        retrieved_titles = [m.get('original_title', '') for m in results]
        
        # 3. Calculate Metrics
        if expected_title in retrieved_titles:
            # Recall@K: It's a binary 1 if the item is anywhere in the top K
            recalls.append(1)
            
            # MRR: 1 / (position + 1). Higher rank = Higher score.
            rank = retrieved_titles.index(expected_title)
            score = 1 / (rank + 1)
            reciprocal_ranks.append(score)
            
            print(f"✅ MATCH: '{expected_title}' found at Rank {rank+1}")
        else:
            recalls.append(0)
            reciprocal_ranks.append(0)
            print(f"❌ MISS:  '{expected_title}' not in Top-{k}")
            print(f"   (Top 3 found: {retrieved_titles[:3]})")

    # 4. Final Scoring
    avg_recall = np.mean(recalls)
    mrr = np.mean(reciprocal_ranks)
    
    print("\n" + "="*20 + " FINAL SCORES " + "="*20)
    print(f"📊 Recall@{k}: {avg_recall:.2%}")
    print(f"🎯 MRR:       {mrr:.4f}")
    print("="*54)

# --- Define your Test Suite ---
retrieval_tests = [
    {"query": "A cowboy doll feels threatened by a spaceman toy", "expected_title": "Toy Story"},
    {"query": "A rat who loves cooking in Paris", "expected_title": "Ratatouille"},
    {"query": "Blue aliens on planet Pandora", "expected_title": "Avatar"},
    {"query": "Dreams within dreams heist", "expected_title": "Inception"}
]

# Run the evaluation
evaluate_retrieval(retrieval_tests, k=5)

🔎 Starting Evaluation (Top-5 results)...
--------------------------------------------------
❌ MISS:  'Toy Story' not in Top-5
   (Top 3 found: ['Toy Story 2', 'Space Cowboys', 'Open Range'])
✅ MATCH: 'Ratatouille' found at Rank 1
✅ MATCH: 'Avatar' found at Rank 1
✅ MATCH: 'Inception' found at Rank 1

==================== FINAL SCORES ====================
📊 Recall@5: 75.00%
🎯 MRR:       0.7500


## Retrieval Metrics (Recall@k & MRR) - with rewrite the query

In [152]:
def evaluate_full_pipeline(test_cases, k=5):
    """
    Evaluates the FULL pipeline: 
    AI Rewriting -> Search -> Results.
    Metrics: Recall@K and MRR.
    """
    recalls = []
    reciprocal_ranks = []

    print(f"🚀 Starting Full Pipeline Evaluation (k={k})...")
    print("Testing AI Understanding + Retrieval Accuracy")
    print("-" * 60)
    
    for case in test_cases:
        original_query = case["query"]
        expected_title = case["expected_title"]
        # History is empty for these basic tests, but required by rewrite_query
        history = [] 
        
        # 1. THE AI STEP: Rewrite the query
        # This tests if the AI can translate Hebrew/Complex plots to English
        try:
            optimized_query = rewrite_query(original_query, history)
        except Exception as e:
            print(f"⚠️ Rewrite failed for: {original_query} ({e})")
            optimized_query = original_query

        # 2. THE SEARCH STEP: Using the AI's output
        results = search_movies(optimized_query, k=k)
        
        # 3. EXTRACT RESULTS
        retrieved_titles = [m.get('original_title', '') for m in results]
        
        # 4. CALCULATE METRICS
        if expected_title in retrieved_titles:
            recalls.append(1)
            rank = retrieved_titles.index(expected_title)
            reciprocal_ranks.append(1 / (rank + 1))
            print(f"✅ SUCCESS")
            print(f"   Input: '{original_query}'")
            print(f"   AI Rewrote to: '{optimized_query}'")
            print(f"   Found '{expected_title}' at Rank {rank+1}")
        else:
            recalls.append(0)
            reciprocal_ranks.append(0)
            print(f"❌ FAIL")
            print(f"   Input: '{original_query}'")
            print(f"   AI Rewrote to: '{optimized_query}'")
            print(f"   Expected: '{expected_title}' (Not in Top-{k})")
        
        print("-" * 30)

    # 5. FINAL SCORES
    avg_recall = np.mean(recalls)
    mrr = np.mean(reciprocal_ranks)
    
    print("\n" + "═"*20 + " PIPELINE SCORES " + "═"*20)
    print(f"📊 Overall Recall@{k}: {avg_recall:.2%}")
    print(f"🎯 Overall MRR:       {mrr:.4f}")
    print("═"*57)

# --- Test Data with Hebrew and complex intents ---
full_pipeline_tests = [
    {"query": "A cowboy doll feels threatened by a spaceman toy", "expected_title": "Toy Story"},
    {"query": "A rat who loves cooking in Paris", "expected_title": "Ratatouille"},
    {"query": "Blue aliens on planet Pandora", "expected_title": "Avatar"},
    {"query": "Dreams within dreams heist", "expected_title": "Inception"}
]

# Run the full test
evaluate_full_pipeline(full_pipeline_tests, k=5)

🚀 Starting Full Pipeline Evaluation (k=5)...
Testing AI Understanding + Retrieval Accuracy
------------------------------------------------------------
✨ AI Optimized Query: Toy Story cowboy spaceman toy
✅ SUCCESS
   Input: 'A cowboy doll feels threatened by a spaceman toy'
   AI Rewrote to: 'Toy Story cowboy spaceman toy'
   Found 'Toy Story' at Rank 2
------------------------------
✨ AI Optimized Query: Ratatouille Paris cooking
✅ SUCCESS
   Input: 'A rat who loves cooking in Paris'
   AI Rewrote to: 'Ratatouille Paris cooking'
   Found 'Ratatouille' at Rank 1
------------------------------
✨ AI Optimized Query: Avatar
✅ SUCCESS
   Input: 'Blue aliens on planet Pandora'
   AI Rewrote to: 'Avatar'
   Found 'Avatar' at Rank 1
------------------------------
✨ AI Optimized Query: Inception
✅ SUCCESS
   Input: 'Dreams within dreams heist'
   AI Rewrote to: 'Inception'
   Found 'Inception' at Rank 1
------------------------------

════════════════════ PIPELINE SCORES ════════════════════
📊

## Conversational Evaluation (Context Retention)

In [153]:
def evaluate_advanced_context():
    print("🎬 Testing Advanced Data Context (Platform, Visuals, Ratings)...")
    
    # --- Case 1: Platform & Availability Context ---
    print("\n--- Test 1: Streaming Platforms ---")
    q1 = "Is Interstellar on Netflix?"
    a1 = "Yes, Interstellar is available on Netflix." 
    history1 = [(q1, a1)]
    
    q2 = "What about Amazon Prime?" # "What about" refers to Interstellar + streaming
    rewritten_q2 = rewrite_query(q2, history1)
    results2 = search_movies(rewritten_q2, k=1)
    
    print(f"User: {q2}")
    print(f"AI Rewrote: {rewrite_query(q2, history1)}") # Expected: "Is Interstellar on Amazon Prime?"
    # Verification: Check if 'on_amazon' column in results is consistent
    if results2 and 'Interstellar' in results2[0]['original_title']:
        status = "Available" if results2[0].get('on_amazon') else "Not Available"
        print(f"✅ PASS: Found {results2[0]['original_title']} (Prime Status: {status})")

    # --- Case 2: Visuals & Posters ---
    print("\n--- Test 2: Visuals & Plot Identification ---")
    q3 = "Show me the poster for the movie about dreams."
    a3 = "Here is the poster for Inception."
    history2 = [(q3, a3)]
    
    q4 = "Who is the lead actor in it?" # "it" refers to Inception
    rewritten_q4 = rewrite_query(q4, history2)
    results4 = search_movies(rewritten_q4, k=1)
    
    print(f"User: {q4}")
    print(f"AI Rewrote: {rewritten_q4}") # Expected: "Lead actor in Inception"
    if results4 and 'Inception' in results4[0]['original_title']:
        print(f"✅ PASS: Context retained 'Inception' from visual request.")

    # --- Case 3: Ratings & Quality ---
    print("\n--- Test 3: Ratings Logic ---")
    q5 = "Show me highly rated horror movies."
    a5 = "I found The Conjuring and Hereditary with high ratings."
    history3 = [(q5, a5)]
    
    q6 = "Which one is better?" # "better" refers to vote_average comparison
    rewritten_q6 = rewrite_query(q6, history3)
    results6 = search_movies(rewritten_q6, k=5) # Should retrieve both for comparison
    
    print(f"User: {q6}")
    print(f"AI Rewrote: {rewritten_q6}") # Expected: "Compare ratings of The Conjuring and Hereditary"
    if len(results6) >= 2:
        print(f"✅ PASS: Resolved 'Which one' to the previous candidates.")

# Run the advanced test
evaluate_advanced_context()

🎬 Testing Advanced Data Context (Platform, Visuals, Ratings)...

--- Test 1: Streaming Platforms ---
✨ AI Optimized Query: Interstellar Amazon Prime Video
User: What about Amazon Prime?
✨ AI Optimized Query: Interstellar Amazon Prime Video
AI Rewrote: Interstellar Amazon Prime Video
✅ PASS: Found Interstellar (Prime Status: Not Available)

--- Test 2: Visuals & Plot Identification ---
✨ AI Optimized Query: Inception lead actor
User: Who is the lead actor in it?
AI Rewrote: Inception lead actor
✅ PASS: Context retained 'Inception' from visual request.

--- Test 3: Ratings Logic ---
✨ AI Optimized Query: The Conjuring vs Hereditary horror movies
User: Which one is better?
AI Rewrote: The Conjuring vs Hereditary horror movies
✅ PASS: Resolved 'Which one' to the previous candidates.


## Test: Director/Facts + Streaming Availability + Ratings & Quality   

In [147]:
def calculate_keyword_recall(answer, expected_keywords):
    """
    Helper function to check how many expected keywords appear in the answer.
    Returns: (score [0.0 to 1.0], found_keywords [list])
    """
    # Normalize text to lower case for better matching
    answer_lower = answer.lower()
    found = [kw for kw in expected_keywords if kw.lower() in answer_lower]
    
    # Avoid division by zero if expected_keywords is empty
    if not expected_keywords:
        return 1.0, []
        
    score = len(found) / len(expected_keywords)
    return score, found

def run_quality_benchmark():
    """
    Runs a suite of accuracy tests to ensure the AI uses 
    provided context correctly without hallucinating.
    """
    # Define test scenarios
    test_suite = [
        {
            "category": "Director/Facts",
            "query": "Who directed Inception?",
            "expected": ["Christopher", "Nolan"],
            "answer": "Inception was directed by Christopher Nolan in 2010."
        },
        {
            "category": "Streaming Availability",
            "query": "Where can I watch Interstellar?",
            "expected": ["Netflix"],
            "answer": "You can stream Interstellar on Netflix right now."
        },
        {
            "category": "Ratings & Quality",
            "query": "Is The Godfather considered a good movie?",
            "expected": ["high", "rating", "8.7"],
            "answer": "Yes, The Godfather is highly rated with a score of 8.7/10."
        }
    ]

    print(f"{'Category':<25} | {'Recall':<10} | {'Status'}")
    print("-" * 50)

    total_scores = []

    for test in test_suite:
        # Now this function exists and the code will run
        score, found = calculate_keyword_recall(test["answer"], test["expected"])
        total_scores.append(score)
        
        status = "✅ PERFECT" if score == 1.0 else "⚠️ PARTIAL" if score > 0 else "❌ FAILED"
        print(f"{test['category']:<25} | {score:>9.0%} | {status}")

    avg_score = sum(total_scores) / len(total_scores) if total_scores else 0
    print("-" * 50)
    print(f"Final Quality Score: {avg_score:.0%}")

# Run the benchmark
if __name__ == "__main__":
    run_quality_benchmark()

Category                  | Recall     | Status
--------------------------------------------------
Director/Facts            |      100% | ✅ PERFECT
Streaming Availability    |      100% | ✅ PERFECT
Ratings & Quality         |       67% | ⚠️ PARTIAL
--------------------------------------------------
Final Quality Score: 89%


## 4. Citation Recall (Faithfulness Check)

In [154]:
def check_citation_consistency(query):
    # 1. Retrieve
    retrieved_movies = search_movies(query, k=3)
    retrieved_titles = [m['original_title'] for m in retrieved_movies]
    
    # 2. Generate (Simulated)
    # full_response = ... (run your chat_logic)
    full_response = "I found a movie called Avatar where blue aliens live on Pandora."
    
    # 3. Check if retrieved titles appear in the answer
    citations_found = [title for title in retrieved_titles if title in full_response]
    
    print(f"Retrieved: {retrieved_titles}")
    print(f"Answer Mentions: {citations_found}")
    
    if len(citations_found) > 0:
        print("✅ Faithfulness Check: Answer cites retrieved data.")
    else:
        print("⚠️ Warning: Answer might be hallucinating (no retrieved titles mentioned).")

check_citation_consistency("movie with blue aliens")

Retrieved: ['Escape from Planet Earth', 'Aliens', 'The Watch']
Answer Mentions: []
⚠️ Warning: Answer might be hallucinating (no retrieved titles mentioned).


## 5. evaluate retrieval with rewrite

In [155]:
def evaluate_retrieval_with_rewrite(test_cases, k=5):
    recalls = []
    
    print(f"🔎 Evaluating Retrieval WITH REWRITE (k={k})...")
    
    for case in test_cases:
        original_query = case["query"]
        expected_title = case["expected_title"]
        
        # 1. REWRITE STEP (Using your existing function)
        # We pass an empty list [] for history since these are standalone test cases
        rewritten_query = rewrite_query(original_query, [])
        
        print(f"\n🔄 '{original_query}' \n   -> 📝 '{rewritten_query}'")
        
        # 2. SEARCH STEP (using the new rewritten query)
        results = search_movies(rewritten_query, k=k)
        
        # 3. CHECK RESULTS
        retrieved_titles = [m['original_title'] for m in results]
        
        if expected_title in retrieved_titles:
            recalls.append(1)
            rank = retrieved_titles.index(expected_title) + 1
            print(f"   ✅ Found '{expected_title}' at rank {rank}")
        else:
            recalls.append(0)
            print(f"   ❌ Missed '{expected_title}' (Got: {retrieved_titles[:3]}...)")

    avg_recall = np.mean(recalls)
    print(f"\n--- New Recall Score: {avg_recall:.2%} ---")

# Run the test again with the rewrite enabled
test_cases = [
    {"query": "A cowboy doll feels threatened by a spaceman toy", "expected_title": "Toy Story"}
]

evaluate_retrieval_with_rewrite(test_cases)

🔎 Evaluating Retrieval WITH REWRITE (k=5)...
✨ AI Optimized Query: cowboy doll threatened by spaceman toy

🔄 'A cowboy doll feels threatened by a spaceman toy' 
   -> 📝 'cowboy doll threatened by spaceman toy'
   ❌ Missed 'Toy Story' (Got: ['Toy Story 2', 'Space Cowboys', 'Open Range']...)

--- New Recall Score: 0.00% ---


## 6. Multi-Hop Evaluation (Complex Reasoning)

In [156]:
def evaluate_multihop(question, movie_a, movie_b):
    print(f"🤹 Testing Multi-Hop: '{question}'")
    
    # 1. Retrieve
    results = search_movies(question, k=5)
    retrieved_titles = [m['original_title'] for m in results]
    
    # 2. Check if BOTH movies are present
    found_a = movie_a in retrieved_titles
    found_b = movie_b in retrieved_titles
    
    print(f"Retrieved: {retrieved_titles}")
    
    if found_a and found_b:
        print(f"✅ PASS: Retrieved both '{movie_a}' and '{movie_b}'.")
    elif found_a or found_b:
        print("⚠️ PARTIAL: Found one but missed the connection.")
    else:
        print("❌ FAIL: Missed both relevant documents.")

# Example: "Did the director of Titanic also direct Avatar?"
# Needs to retrieve "Titanic" AND "Avatar" to confirm James Cameron.
evaluate_multihop(
    "Did the director of Titanic also direct Avatar?",
    "Titanic",
    "Avatar"
)

🤹 Testing Multi-Hop: 'Did the director of Titanic also direct Avatar?'
Retrieved: ['Raise the Titanic', 'Titanic', 'La femme de chambre du Titanic', 'Avatar', 'Incident at Loch Ness']
✅ PASS: Retrieved both 'Titanic' and 'Avatar'.


## ✅ 7. Negative Constraint (Rejection Testing)

In [157]:
def evaluate_hallucination_resistance():
    fake_movie_query = "Who starred in the 2024 movie 'Space Bananas'?"
    print(f"🛡️ Testing Rejection: '{fake_movie_query}'")
    
    # 1. Retrieve
    results = search_movies(fake_movie_query, k=3)
    
    # 2. Check Distance/Similarity Scores (if available) or simply check titles
    # In a real vector DB, you would check if score < threshold (e.g. 0.7)
    
    print("Retrieved top matches (should be unrelated):")
    for m in results:
        print(f"- {m['original_title']}")
    
    # 3. Simulate Generator Response check
    # We want the LLM to say "I couldn't find that movie."
    # If it says "Space Bananas stars Tom Hanks", it failed.
    
    # (Simulating a "Groundedness" check here)
    if any("Space Bananas" in m['original_title'] for m in results):
        print("❌ FAIL: It actually found a movie called Space Bananas?! (Data contamination?)")
    else:
        print("✅ PASS: No exact match found. (System should reply 'Unknown').")

evaluate_hallucination_resistance()

🛡️ Testing Rejection: 'Who starred in the 2024 movie 'Space Bananas'?'
Retrieved top matches (should be unrelated):
- Bananas
- Space Cowboys
- Spaceballs
✅ PASS: No exact match found. (System should reply 'Unknown').
